# Finalize Anima Qwen3.5-0.8B Encoder — v3 + v4 stability
Convert the current v2 bridge/profile into the one-file v3 final encoder while enabling v4 token-energy and binding-preservation defaults.


In [ ]:
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'scripts' / 'finalize_anima_text_encoder.py').exists() and (REPO_ROOT.parent / 'scripts' / 'finalize_anima_text_encoder.py').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'src'))
print(REPO_ROOT)


In [ ]:
BRIDGE_PROFILE = '/workspace/qwen35_08b_to_qwen3_06b.profile.safetensors'
SOURCE_MODEL = '/workspace/qwen3.5-0.8b-base.safetensors'  # set None if profile already bundles encoder.*
OUTPUT = '/workspace/Anima-Qwen3.5-0.8B-Final.safetensors'

# Capability-retaining semantic expansion.
EXPANSION_STRENGTH = 0.25
EXPANSION_MAX_TOKENS = 16
EXPANSION_CHUNK_SIZE = 64
EXPANSION_MIN_SOURCE_TOKENS = 16
EXPANSION_RESIDUAL_CLIP = 0.30
EXPANSION_GROUP_AWARE = True
EXPANSION_COHERENCE_POWER = 1.0
EXPANSION_MIN_COHERENCE = 0.15

# v4 primary-memory stability. Keep calibration metrics in the profile, but use
# image-runtime settings that preserve token separation and avoid energy spikes.
FINAL_CENTER_STRENGTH = 1.0
FINAL_VARIANCE_STRENGTH = 0.0
FINAL_RMS_STRENGTH = 0.0
FINAL_DELTA_CLIP_RATIO = 0.30
FINAL_TOKEN_RMS_STRENGTH = 0.70
FINAL_TOKEN_RMS_MIN_RATIO = 0.85
FINAL_TOKEN_RMS_MAX_RATIO = 1.10


In [ ]:
from finalize_anima_text_encoder import FinalEncoderConfig, finalize_anima_text_encoder
result = finalize_anima_text_encoder(FinalEncoderConfig(
    bridge_profile=BRIDGE_PROFILE, source_model=SOURCE_MODEL, output=OUTPUT,
    semantic_expansion_strength=EXPANSION_STRENGTH,
    semantic_expansion_max_tokens=EXPANSION_MAX_TOKENS,
    semantic_expansion_chunk_size=EXPANSION_CHUNK_SIZE,
    semantic_expansion_min_source_tokens=EXPANSION_MIN_SOURCE_TOKENS,
    semantic_expansion_residual_clip=EXPANSION_RESIDUAL_CLIP,
    semantic_expansion_group_aware=EXPANSION_GROUP_AWARE,
    semantic_expansion_coherence_power=EXPANSION_COHERENCE_POWER,
    semantic_expansion_min_coherence=EXPANSION_MIN_COHERENCE,
    final_center_strength=FINAL_CENTER_STRENGTH,
    final_variance_strength=FINAL_VARIANCE_STRENGTH,
    final_rms_strength=FINAL_RMS_STRENGTH,
    final_delta_clip_ratio=FINAL_DELTA_CLIP_RATIO,
    final_token_rms_strength=FINAL_TOKEN_RMS_STRENGTH,
    final_token_rms_min_ratio=FINAL_TOKEN_RMS_MIN_RATIO,
    final_token_rms_max_ratio=FINAL_TOKEN_RMS_MAX_RATIO,
))
print(result.summary())


## Runtime A/B
The final file is still `anima_text_encoder_v3`; v4 is an incremental stability layer, not a second encoder format. Start with the stored v4 defaults. For diagnostics, use `pipe.set_text_encoder_conditioning_stability(...)` and `pipe.set_semantic_expansion(...)`. Semicolon/BREAK/AND PromptPlan groups are kept separate for semantic summary slots.
